In [9]:
import os

java_home = "/home/bruno/.jdk/jdk-17.0.19+10"
os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = f"{java_home}/bin:" + os.environ["PATH"]

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local")
    .appName("PySpark_01")
    .getOrCreate()
)
print(spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/20 17:21:31 WARN Utils: Your hostname, bruno-B550M-AORUS-ELITE, resolves to a loopback address: 127.0.1.1; using 192.168.4.2 instead (on interface enp4s0)
26/06/20 17:21:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/20 17:21:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


4.1.2


In [10]:
from pyspark.sql import functions as F
import re

In [11]:
spark = (
    SparkSession.builder
    .master("local")
    .appName("PySpark_01")
    .getOrCreate()
)

print(spark.version)



4.1.2


In [12]:
df = spark.read.parquet("/home/bruno/projeto_clima-df/data/raw_df/INMET_CO_DF_A042_BRAZLANDIA_01-01-2025_A_31-12-2025.parquet", header=True, inferSchema=True)
df.show(3)

26/06/20 17:21:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------+---+----------+------------+------------+-----------+--------+----------------+----------+--------+--------------------------------+-----------------------------------------------------+-----------------------------------------------+------------------------------------------------+-----------------------+--------------------------------------------+------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------------+------------------------------------------------+----------------------------------------+----------------------------------------+-----------------------------------+------------------------------------+--------------------------+-------------------------------+
|REGIAO| UF|   ESTACAO|CODIGO (WMO)|    LATITUDE|  LONGITUDE|ALTITUDE|DATA DE FUNDACAO|      Data|Hora UTC|PRECIPITAÇÃO TOTAL, HORÁRIO (mm)|PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)|PRESSÃO AT

### Removendo o "." do nome das colunas

In [13]:
df = df.toDF(*[col.replace(".", "") for col in df.columns])

### Excluindo Colunas Desnecessárias 

In [14]:
remover_colunas = ["CODIGO (WMO)", "DATA DE FUNDACAO"]
df = df.drop(*remover_colunas)

In [15]:
df.printSchema()

root
 |-- REGIAO: string (nullable = true)
 |-- UF: string (nullable = true)
 |-- ESTACAO: string (nullable = true)
 |-- LATITUDE: string (nullable = true)
 |-- LONGITUDE: string (nullable = true)
 |-- ALTITUDE: string (nullable = true)
 |-- Data: string (nullable = true)
 |-- Hora UTC: string (nullable = true)
 |-- PRECIPITAÇÃO TOTAL, HORÁRIO (mm): double (nullable = true)
 |-- PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB): double (nullable = true)
 |-- PRESSÃO ATMOSFERICA MAXNA HORA ANT (AUT) (mB): double (nullable = true)
 |-- PRESSÃO ATMOSFERICA MIN NA HORA ANT (AUT) (mB): double (nullable = true)
 |-- RADIACAO GLOBAL (Kj/m²): double (nullable = true)
 |-- TEMPERATURA DO AR - BULBO SECO, HORARIA (°C): double (nullable = true)
 |-- TEMPERATURA DO PONTO DE ORVALHO (°C): double (nullable = true)
 |-- TEMPERATURA MÁXIMA NA HORA ANT (AUT) (°C): double (nullable = true)
 |-- TEMPERATURA MÍNIMA NA HORA ANT (AUT) (°C): double (nullable = true)
 |-- TEMPERATURA ORVALHO MAX NA HORA A

### Renomeando Colunas

In [ ]:
padronizar_colunas = [
    col.lower().replace(",","_").replace(" ", "_")
    for col in df.columns
]

renames = {
    "precipitação_total__horário_(mm)": "precipitacao_total_mm",
    "pressao_atmosferica_ao_nivel_da_estacao__horaria_(mb)": "pressao_estacao_mb",
    "pressão_atmosferica_maxna_hora_ant_(aut)_(mb)": "pressao_max_mb",
    "pressão_atmosferica_min_na_hora_ant_(aut)_(mb)": "pressao_min_mb",
    "radiacao_global_(kj/m²)": "radiacao_global_kj_m2",
    "temperatura_do_ar_-_bulbo_seco__horaria_(°c)": "temperatura_seco_c",
    "temperatura_do_ponto_de_orvalho_(°c)": "temperatura_orvalho_c",
    "temperatura_máxima_na_hora_ant_(aut)_(°c)": "temperatura_max_c",
    "temperatura_mínima_na_hora_ant_(aut)_(°c)": "temperatura_min_c",
    "temperatura_orvalho_max_na_hora_ant_(aut)_(°c)": "temperatura_orvalho_max_c",
    "temperatura_orvalho_min_na_hora_ant_(aut)_(°c)": "temperatura_orvalho_min_c",
    "umidade_rel_max_na_hora_ant_(aut)_(%)": "umidade_max_porcento",
    "umidade_rel_min_na_hora_ant_(aut)_(%)": "umidade_min_porcento",
    "umidade_relativa_do_ar__horaria_(%)": "umidade_porcento",
    "vento__direção_horaria_(gr)_(°_(gr))": "vento_direcao_graus",
    "vento__rajada_maxima_(m/s)": "vento_rajada_ms",
    "vento__velocidade_horaria_(m/s)": "vento_velocidade_ms"
}


df = df.toDF(*padronizar_colunas)
for antiga_col, nova_col in renames.items():
    df = df.withColumnRenamed(antiga_col, nova_col)
    
df.columns

['regiao',
 'uf',
 'estacao',
 'latitude',
 'longitude',
 'altitude',
 'data',
 'hora_utc',
 'precipitacao_total_mm',
 'pressao_estacao_mb',
 'pressao_max_mb',
 'pressao_min_mb',
 'radiacao_global_kj_m2',
 'temperatura_seco_c',
 'temperatura_orvalho_c',
 'temperatura_max_c',
 'temperatura_min_c',
 'temperatura_orvalho_max_c',
 'temperatura_orvalho_min_c',
 'umidade_max_porcento',
 'umidade_min_porcento',
 'umidade_porcento',
 'vento_direcao_graus',
 'vento_rajada_ms',
 'vento_velocidade_ms']

### Verificando nulos

In [ ]:
for coluna in df.columns:
    print(coluna, df.filter(df[coluna].isNull()).count())

REGIAO 0
UF 0
ESTACAO 0
CODIGO (WMO) 0
LATITUDE 0
LONGITUDE 0
ALTITUDE 0
DATA DE FUNDACAO 0
Data 0
Hora UTC 0
PRECIPITAÇÃO TOTAL, HORÁRIO (mm) 14
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB) 13
PRESSÃO ATMOSFERICA MAXNA HORA ANT (AUT) (mB) 16
PRESSÃO ATMOSFERICA MIN NA HORA ANT (AUT) (mB) 17
RADIACAO GLOBAL (Kj/m²) 4040
TEMPERATURA DO AR - BULBO SECO, HORARIA (°C) 13
TEMPERATURA DO PONTO DE ORVALHO (°C) 13
TEMPERATURA MÁXIMA NA HORA ANT (AUT) (°C) 16
TEMPERATURA MÍNIMA NA HORA ANT (AUT) (°C) 16
TEMPERATURA ORVALHO MAX NA HORA ANT (AUT) (°C) 16
TEMPERATURA ORVALHO MIN NA HORA ANT (AUT) (°C) 17
UMIDADE REL MAX NA HORA ANT (AUT) (%) 16
UMIDADE REL MIN NA HORA ANT (AUT) (%) 16
UMIDADE RELATIVA DO AR, HORARIA (%) 13
VENTO, DIREÇÃO HORARIA (gr) (° (gr)) 104
VENTO, RAJADA MAXIMA (m/s) 106
VENTO, VELOCIDADE HORARIA (m/s) 105


In [ ]:
df.describe().show()

+-------+------+----+----------+------------+------------+-----------+--------+----------------+----------+--------+--------------------------------+-----------------------------------------------------+---------------------------------------------+----------------------------------------------+-----------------------+--------------------------------------------+------------------------------------+-----------------------------------------+-----------------------------------------+----------------------------------------------+----------------------------------------------+-------------------------------------+-------------------------------------+-----------------------------------+------------------------------------+--------------------------+-------------------------------+
|summary|REGIAO|  UF|   ESTACAO|CODIGO (WMO)|    LATITUDE|  LONGITUDE|ALTITUDE|DATA DE FUNDACAO|      Data|Hora UTC|PRECIPITAÇÃO TOTAL, HORÁRIO (mm)|PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)|PRESSÃO 